### CAP 5619: AI in FinTech
+ Fall 2026
+ Hubert Antoniewicz, hu827797
+ Submission Date: November 2026
+ Homework: Bitcoin Price Prediction with Explainable AI

Predicts Bitcoin's daily price for December 2017 (one-day-ahead) from occurrence (OCC) and amount (AMO) chainlet features plus lagged prices, then explains the model with SHAP.

# Bitcoin Price Prediction with Chainlets and SHAP

### Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from sklearn.linear_model import Ridge, Lasso
import shap

SEED = 100
np.random.seed(SEED)
tf.random.set_seed(SEED)
pd.set_option('display.precision', 4)

DATA_DIR = 'data'

## 1. Load Data and Parse Dates

Price, occurrence (OCC) and amount (AMO) chainlet files are loaded as-is. Dates are built with datetime functions only:
+ price file: `date` parsed with an explicit format (`%m/%d/%Y %H:%M`), so month/day order can never be guessed wrong;
+ chainlet files: `year` + 1-based day-of-year converted to a date with datetime arithmetic (Jan 1 + (day - 1) days).

In [2]:
price = pd.read_csv(f'{DATA_DIR}/pricedBitcoin2009-2018.csv')
occ = pd.read_csv(f'{DATA_DIR}/OccChainletsInTime.txt', sep='\t')
amo = pd.read_csv(f'{DATA_DIR}/AmoChainletsInTime.txt', sep='\t')
price.shape, occ.shape, amo.shape

((2404, 5), (3281, 403), (3281, 403))

In [3]:
def day_of_year_to_date(year, day):
    """Convert (year, 1-based day-of-year) columns to datetimes."""
    jan1 = pd.to_datetime(pd.DataFrame({'year': year, 'month': 1, 'day': 1}))
    return jan1 + pd.to_timedelta(day - 1, unit='D')

price['date'] = pd.to_datetime(price['date'], format='%m/%d/%Y %H:%M')
# concat (not column assignment) avoids fragmenting the 403-column frames.
occ = pd.concat([day_of_year_to_date(occ['year'], occ['day']).rename('date'), occ], axis=1)
amo = pd.concat([day_of_year_to_date(amo['year'], amo['day']).rename('date'), amo], axis=1)

price.head()

,date,price,year,day,totaltx
0,2011-06-01,9.57,2011,152,6464
1,2011-06-02,10.60,2011,153,7632
2,2011-06-03,14.29,2011,154,8502
3,2011-06-04,18.89,2011,155,8766
4,2011-06-05,16.70,2011,156,8675


In [4]:
occ.iloc[:3, :6]

,date,year,day,totaltx,1:1,1:2
0,2009-01-03,2009,3,1,1,0
1,2009-01-08,2009,8,14,14,0
2,2009-01-09,2009,9,7,7,0


### Sanity Checks

Each check fails loudly if the data is not what the rest of the notebook assumes.

In [5]:
# Both date sources must agree, otherwise the merge would misalign days.
assert (price['date'].dt.year == price['year']).all()
assert (price['date'].dt.dayofyear == price['day']).all()

for name, d in [('price', price), ('occ', occ), ('amo', amo)]:
    assert d.isna().sum().sum() == 0, f'{name}: missing values'
    assert d['date'].is_unique and d['date'].is_monotonic_increasing, f'{name}: dates not unique/sorted'
    assert (d.select_dtypes('number') >= 0).all().all(), f'{name}: negative values'

assert list(occ.columns) == list(amo.columns)
assert occ['date'].equals(amo['date'])

# OCC totaltx is the transaction count (= sum of chainlet counts);
# AMO totaltx is the total satoshis moved, so it cannot be a join key with price.
chainlet_cols = [c for c in occ.columns if ':' in c]
assert len(chainlet_cols) == 400
assert (occ[chainlet_cols].sum(axis=1) == occ['totaltx']).all()

dec_start, dec_end = pd.Timestamp(2017, 12, 1), pd.Timestamp(2017, 12, 31)
for name, d in [('price', price), ('occ', occ), ('amo', amo)]:
    assert d['date'].between(dec_start, dec_end).sum() == 31, f'{name}: December 2017 incomplete'

pd.DataFrame({name: [d['date'].min().date(), d['date'].max().date(), len(d)]
              for name, d in [('price', price), ('occ', occ), ('amo', amo)]},
             index=['first day', 'last day', 'rows'])

,price,occ,amo
first day,2011-06-01,2009-01-03,2009-01-03
last day,2017-12-31,2017-12-31,2017-12-31
rows,2404,3281,3281


In [6]:
# Calendar days missing from the price series (gaps > 1 day between rows).
full_range = pd.date_range(price['date'].min(), price['date'].max(), freq='D')
full_range.difference(price['date'])

DatetimeIndex(['2012-12-31', '2016-12-31'], dtype='datetime64[us]', freq=None)

All checks pass. Findings:
+ The price series covers 2011-06-01 to 2017-12-31; chainlets start in 2009 and end on the same day.
+ All 31 days of December 2017 are present in all three files.
+ The price file has no rows for 2012-12-31 and 2016-12-31 (day 366 of leap years). Lags are therefore built on a daily calendar (Section 3), not by row position.
+ AMO's `totaltx` column holds total satoshis, not a transaction count, so files are merged on `date` (Section 2).

## 2. Merge OCC, AMO, and Price

Chainlet columns are prefixed (`occ_`, `amo_`) so both feature families can live in one frame. The two chainlet files are joined first, then price is left-joined onto that daily calendar: the two days missing from the price file stay as rows with no price, so that later shifts move by exactly one calendar day.

`year` and `day` are dropped here; `date` is kept only for splitting and is never a model input.

In [7]:
occ_feat = occ[['date', 'totaltx'] + chainlet_cols].rename(columns={c: f'occ_{c}' for c in chainlet_cols})
amo_feat = (amo[['date', 'totaltx'] + chainlet_cols]
            .rename(columns={'totaltx': 'amo_total', **{c: f'amo_{c}' for c in chainlet_cols}}))

chain = occ_feat.merge(amo_feat, on='date', how='inner', validate='one_to_one')
df = chain.merge(price[['date', 'price', 'totaltx']], on='date', how='left',
                 suffixes=('', '_price'), validate='one_to_one')

# Price and OCC must report the same transaction count on every priced day.
priced = df['price'].notna()
assert (df.loc[priced, 'totaltx'] == df.loc[priced, 'totaltx_price']).all()
assert priced.sum() == len(price)
df = df.drop(columns='totaltx_price')

# Keep the span covered by price data; it must be one continuous daily calendar.
df = df[df['date'] >= price['date'].min()].reset_index(drop=True)
assert (df['date'].diff().dropna() == pd.Timedelta(days=1)).all()

occ_cols = [f'occ_{c}' for c in chainlet_cols]
amo_cols = [f'amo_{c}' for c in chainlet_cols]
df.shape

(2406, 804)

In [8]:
df.loc[df['price'].isna(), ['date', 'price', 'totaltx', 'amo_total']]

,date,price,totaltx,amo_total
579,2012-12-31,NaN,27844,136824068787140
2040,2016-12-31,NaN,224275,119964952102904


In [9]:
df[['date', 'price', 'totaltx', 'amo_total', 'occ_1:1', 'amo_1:1', 'occ_20:20', 'amo_20:20']].tail()

,date,price,totaltx,amo_total,occ_1:1,amo_1:1,occ_20:20,amo_20:20
2401,2017-12-27,15589.3217,316216,236518206836146,21490,2291875556485,601,7555792733727
2402,2017-12-28,14380.5817,298345,206446934246827,19408,1814758242922,485,6286839224749
2403,2017-12-29,14640.1400,353859,229190878415673,21426,1757501880496,479,6619277402266
2404,2017-12-30,13215.5740,350328,217660764650622,23026,2079904998918,519,9230771994520
2405,2017-12-31,14165.5750,262815,161105714128028,18885,1527068219486,288,4490695347545


The merged frame has one row per calendar day from 2011-06-01 to 2017-12-31 (2,406 days): the 2,404 priced days plus the two leap-year days without a price. Each row holds the price, the transaction count (`totaltx`), the total satoshis moved (`amo_total`), and 400 OCC plus 400 AMO chainlet columns.

## 3. Feature Engineering

Each row describes one target day *t*. Every feature uses only information available by the end of day *t-1*, which keeps the horizon at exactly one day.

+ **Price lags** `price_lag1..K`: prices on days *t-1 ... t-K* (`.shift(k)`; the frame is a continuous daily calendar, so one row = one day).
+ **Lag summaries**: `mom` (mean day-over-day change across the lags), `vol` (std of those changes), `range_pos` (where yesterday's price sits between the K-day min and max).
+ **Chainlets** (OCC and AMO) plus `totaltx` and `amo_total`, all from day *t-1* and `log1p`-transformed: AMO values reach ~10^15 satoshis and both families are heavily right-skewed.
+ **Target**: the price change `price_t - price_{t-1}`. The predicted price is `price_lag1 + predicted change`, and RMSE is always computed on that reconstructed price in USD.

Rows whose lags touch the two days without a price are dropped rather than filled. Interpolation would use the next day's price, which is leakage.

In [10]:
def build_features(data, n_lags):
    """One row per target day t; every feature uses only data up to day t-1."""
    lags = pd.concat({f'price_lag{k}': data['price'].shift(k) for k in range(1, n_lags + 1)}, axis=1)

    # Day-over-day changes inside the lag window (all strictly before t).
    changes = lags.iloc[:, :-1].values - lags.iloc[:, 1:].values
    lo, hi = lags.min(axis=1, skipna=False), lags.max(axis=1, skipna=False)
    summary = pd.DataFrame({
        'mom': changes.mean(axis=1),
        'vol': changes.std(axis=1),
        'range_pos': (lags['price_lag1'] - lo) / (hi - lo).replace(0, np.nan),
    }, index=data.index)

    chain = np.log1p(data[['totaltx', 'amo_total'] + occ_cols + amo_cols].shift(1)).add_suffix('_lag1')

    target = (data['price'] - lags['price_lag1']).rename('target')
    out = pd.concat([data[['date', 'price']], lags, summary, chain, target], axis=1)
    return out.dropna().reset_index(drop=True)

def feature_columns(frame):
    return [c for c in frame.columns if c not in ('date', 'price', 'target')]

In [11]:
feat = build_features(df, n_lags=7)
feat_cols = feature_columns(feat)

assert not {'year', 'day', 'date'} & set(feat_cols)
assert feat['date'].between(pd.Timestamp(2017, 12, 1), pd.Timestamp(2017, 12, 31)).sum() == 31
assert np.allclose(feat['price_lag1'] + feat['target'], feat['price'])

print(f'{len(feat)} rows, {len(feat_cols)} features '
      f"({sum(c.startswith('occ_') for c in feat_cols)} OCC chainlets, "
      f"{sum(c.startswith('amo_') and ':' in c for c in feat_cols)} AMO chainlets)")
feat[['date', 'price', 'target', 'price_lag1', 'price_lag2', 'mom', 'vol', 'range_pos',
      'totaltx_lag1', 'occ_1:1_lag1', 'amo_1:1_lag1']].tail()

2382 rows, 812 features (400 OCC chainlets, 400 AMO chainlets)


,date,price,target,price_lag1,price_lag2,mom,vol,range_pos,totaltx_lag1,occ_1:1_lag1,amo_1:1_lag1
2377,2017-12-27,15589.3217,-409.7267,15999.0483,14119.0283,-4.5372,1026.0921,0.9769,12.5398,9.8132,27.8121
2378,2017-12-28,14380.5817,-1208.7400,15589.3217,15999.0483,-76.3647,1036.8019,0.7816,12.6642,9.9754,28.4604
2379,2017-12-29,14640.1400,259.5583,14380.5817,15589.3217,-135.0606,1088.0134,0.2105,12.6060,9.8735,28.2270
2380,2017-12-30,13215.5740,-1424.5660,14640.1400,14380.5817,-120.0203,1092.7306,0.3371,12.7767,9.9724,28.1949
2381,2017-12-31,14165.5750,950.0010,13215.5740,14640.1400,-122.2668,1095.3932,0.0000,12.7666,10.0444,28.3633


### Leakage Check

Features for a December day must not change when every later day is removed and that day's own price is hidden. If any feature looked at day *t* or later, the two rows below would differ.

In [12]:
def assert_no_lookahead(data, day, n_lags):
    visible = data[data['date'] <= day].copy()
    visible.loc[visible['date'] == day, 'price'] = 12345.0  # dummy: the true price is unknown at prediction time
    cols = feature_columns(build_features(data, n_lags))
    full_row = build_features(data, n_lags).set_index('date').loc[day, cols]
    past_row = build_features(visible, n_lags).set_index('date').loc[day, cols]
    pd.testing.assert_series_equal(full_row, past_row)

for day in pd.date_range('2017-12-01', '2017-12-31', freq='D'):
    assert_no_lookahead(df, day, n_lags=7)
print('Features for all 31 December days use only past data.')

Features for all 31 December days use only past data.


The feature table has 2,382 daily rows (2011-06 to 2017-12, minus rows whose lags touch a missing price) and 812 candidate features: 7 price lags, 3 lag summaries, `totaltx`, `amo_total`, and 400 OCC plus 400 AMO chainlets. The number of lags is tuned later, and the chainlet columns are narrowed down using training data only (Section 4).

## 4. Chronological Split and Scaling

## 5. Model Selection

## 6. Final Model and December 2017 Evaluation

## 7. Explaining the Model with SHAP

## 8. Save Predictions